In [23]:
!pip install --upgrade pyproject-hooks build


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [24]:
!pip install --use-pep517 kaggle


[notice] A new release of pip is available: 24.3.1 -> 25.0.1
[notice] To update, run: pip install --upgrade pip


In [25]:
import os

# Crea la carpeta .kaggle en el home si no existe
os.makedirs(os.path.expanduser("~/.kaggle"), exist_ok=True)

# Mueve el kaggle.json subido a ~/.kaggle/kaggle.json
!mv kaggle.json ~/.kaggle/kaggle.json

# Ajusta permisos a 600
!chmod 600 ~/.kaggle/kaggle.json

print("kaggle.json configurado en ~/.kaggle/")


kaggle.json configurado en ~/.kaggle/


In [26]:
!rm -rf /tmp/data/*
!echo "✅ Archivos antiguos eliminados."


✅ Archivos antiguos eliminados.


In [34]:
# 1️⃣ Crear la carpeta de trabajo
!mkdir -p /tmp/data

# 2️⃣ Descargar el dataset completo
!kaggle datasets download -d marcosdiezz/consumo_electrico/0 -p /tmp/data

# 3️⃣ Descomprimir el archivo deseado (ajústalo según el nuevo nombre del archivo)
!unzip -o "/tmp/data/Consumo_Electrico_Timestamp.zip" "divididof/consumo_parte_116.csv" -d /tmp/data

# 4️⃣ Mover el archivo al directorio raíz
!mv /tmp/data/divididof/consumo_parte_116.csv /tmp/data/

# 5️⃣ Eliminar archivos no deseados para limpiar espacio
!rm -rf /tmp/data/divididof /tmp/data/Consumo_Electrico_Timestamp.zip

# 6️⃣ Verificar que solo queda el archivo correcto
!ls -lh /tmp/data


403 - Forbidden - Permission 'datasets.get' was denied
unzip:  cannot find or open /tmp/data/Consumo_Electrico_Timestamp.zip, /tmp/data/Consumo_Electrico_Timestamp.zip.zip or /tmp/data/Consumo_Electrico_Timestamp.zip.ZIP.
mv: cannot stat '/tmp/data/divididof/consumo_parte_116.csv': No such file or directory
total 0


In [40]:
import mlrun
import pandas as pd

# Ruta del dataset seleccionado
csv_path = "consumo_ts_parte_0.csv"

# Cargar el CSV en un DataFrame
df = pd.read_csv(csv_path)

project = mlrun.get_or_create_project(
    name="smartgrids",
    context="./smartgrids"
)

# Loggear el dataset en MLRun para que se guarde en MinIO
project.log_dataset("consumo_electrico", df=df, format="csv")

print("✅ Dataset ",csv_path," subido a MLRun y MinIO.")

> 2025-03-10 17:25:02,194 [info] Project loaded successfully: {"project_name":"smartgrids"}
✅ Dataset  consumo_ts_parte_0.csv  subido a MLRun y MinIO.


In [41]:
%%writefile preprocess.py
import mlrun
from storey import MapClass
import pandas as pd

def preprocess_data(context, file_path: str):
    """Preprocesa y registra datos en el Feature Store."""
    
    # Cargar el dataset
    df = pd.read_csv(file_path)

    # Eliminar columnas originales de fecha y hora
    df['timestamp'] = pd.to_datetime(df['timestamp'])

    # Eliminar filas con valores nulos
    df.dropna(inplace=True)

    # Filtrar valores negativos en consumo_kwh y coste_euros
    df = df[(df["consumo_kwh"] > 0) & (df["coste_euros"] > 0)]

    # Obtener el proyecto de MLRun
    project = mlrun.get_or_create_project("smartgrids")

    # Definir el Feature Store
    feature_set = project.get_feature_set(
        name="consumo_electrico",
        entities=["casa_id", "fecha_hora"],  # Llaves primarias
        description="Consumo eléctrico por casa y tiempo"
    )

    # Ingerir los datos en el Feature Store
    feature_set.ingest(df)

    # Guardar dataset preprocesado en Parquet
    context.log_dataset("consumo_electrico", df=df, format="parquet")

    print("✅ Datos preprocesados y almacenados en el Feature Store.")

Overwriting preprocess.py


In [15]:
import mlrun
from mlrun import code_to_function

# Crear el proyecto en MLRun
project = mlrun.get_or_create_project("smartgrids", context="./")

# Registrar la función de preprocesamiento
preprocess_func = code_to_function(
    name="preprocess-data",
    filename="preprocess.py",
    handler="preprocess_data",
    kind="job",
    image="mlrun/mlrun"
)

project.set_function(preprocess_func)
project.save()

> 2025-03-10 11:01:05,444 [info] Project loaded successfully: {"project_name":"smartgrids"}


In [16]:
run = preprocess_func.run(
    inputs={"file_path": "s3://mlrun/preprocess-data-preprocess-data/0/consumo_electrico_preprocesado.csv"},  # Se usa la ruta de MinIO obtenida desde MLRun
    local=True
)

> 2025-03-10 11:01:09,563 [warning] it is recommended to use k8s secret (specify secret_name), specifying the aws_access_key/aws_secret_key directly is unsafe
> 2025-03-10 11:01:09,566 [info] Storing function: {"db":null,"name":"preprocess-data-preprocess-data","uid":"8ebf210d9a1540fd805464d0c21c721c"}
> 2025-03-10 11:01:09,642 [info] downloading s3://mlrun/preprocess-data-preprocess-data/0/consumo_electrico_preprocesado.csv to local temp file
> 2025-03-10 11:01:10,254 [error] Execution error, Traceback (most recent call last):
  File "/opt/conda/lib/python3.9/site-packages/mlrun/runtimes/local.py", line 506, in exec_from_params
    val = mlrun.handler(
  File "/opt/conda/lib/python3.9/site-packages/mlrun/package/__init__.py", line 140, in wrapper
    func_outputs = func(*args, **kwargs)
  File "preprocess.py", line 12, in preprocess_data
    df["fecha_hora"] = pd.to_datetime(df["fecha"] + " " + df["hora"].astype(str))
  File "/opt/conda/lib/python3.9/site-packages/pandas/core/tools/da

Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
Unknown datetime string format, unable to parse: 2020-10-27 1, at position 0


project,uid,iter,start,state,kind,name,labels,inputs,parameters,results
smartgrids,...c21c721c,0,Mar 10 11:01:09,error,run,preprocess-data-preprocess-data,kind=localowner=jovyanhost=mlrun-jupyter-d8f976ff-6lzgm,file_path,,


> 2025-03-10 11:01:10,330 [info] Run execution finished: {"name":"preprocess-data-preprocess-data","status":"error"}


RunError: Unknown datetime string format, unable to parse: 2020-10-27 1, at position 0

In [ ]:
run = preprocess_func.run(
    inputs={"file_path": mlrun.get_dataitem("store://datasets/consumo_electrico").get_target_path()},
    local=True
)